## Image Rotation

In [ ]:
# import os
# import cv2
# from concurrent.futures import ThreadPoolExecutor
# from tqdm import tqdm

# def rotate_image(img_path):
#     """Rotate a single image 90 degrees counter-clockwise"""
#     img = cv2.imread(img_path)
#     if img is None:
#         return f"Failed: {img_path}"
    
#     # OpenCV rotate 90 degrees counter-clockwise
#     rotated = cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)
    
#     # Get filename and create output path
#     filename = os.path.basename(img_path)
#     output_path = os.path.join("img/segments_rotated", filename)
    
#     cv2.imwrite(output_path, rotated)
#     return f"Rotated: {filename}"

# # Get list of image files from input directory
# input_dir = "img/segments"
# image_extensions = ('.png', '.jpg', '.jpeg', '.gif', '.bmp')

# image_files = [
#     os.path.join(input_dir, f) 
#     for f in os.listdir(input_dir) 
#     if f.lower().endswith(image_extensions)
# ]

# # Create output directory if it doesn't exist
# os.makedirs("img/segments_rotated", exist_ok=True)

# # Process images in parallel using ThreadPoolExecutor (works in Jupyter)
# with ThreadPoolExecutor(max_workers=4) as executor:
#     results = tqdm(executor.map(rotate_image, image_files), total=len(image_files), desc="Rotating images")

# for result in results:
#     print(result)

# print(f"All {len(image_files)} images rotated successfully!")

## Review File Generator

In [3]:
import pandas as pd

df = pd.read_csv('spine_extraction_results_incremental.csv')
df.file = "http://localhost:8000/" + df.file
df.to_csv('spine_extraction_results_incremental_full_url.csv', index=False)
df

,file,title,author,call_no
0,http://localhost:8000/03d4dc70-IMG_2817_spine_...,NEWLY INDUSTRIALIZING COUNTRIES AND THE POLITI...,ED BY JERGER CARLESON AND TIMOTHY M. SHAW,HF 1413 .N497 1988
1,http://localhost:8000/03d4dc70-IMG_2817_spine_...,Fair Trade For All,Stiglitz and Charlton,HF 1413 $85 2005
2,http://localhost:8000/03d4dc70-IMG_2817_spine_...,Third World Strategy,Gauhar,HF 1413 T5 1983
3,http://localhost:8000/03d4dc70-IMG_2817_spine_...,U.S. FOREIGN POLICY AND THE THIRD WORLD: AGEND...,Lewis Kallab,HF 1413 U5x 1983
4,http://localhost:8000/03d4dc70-IMG_2817_spine_...,U.S. FOREIGN POLICY and the THIRD WORLD: AGEND...,NaN,HF 1413 U535 1985
...,...,...,...,...
2727,http://localhost:8000/e582243e-IMG_3614_spine_...,MEET THE PRESS,KRAUS,E 743 M4
2728,http://localhost:8000/e582243e-IMG_3614_spine_...,MEET THE PRESS,12,E 743 M4
2729,http://localhost:8000/e582243e-IMG_3614_spine_...,MEET THE PRESS,13,E 743 M4
2730,http://localhost:8000/e582243e-IMG_3614_spine_...,MEET THE PRESS,KRAUS,E 743 H4


## Training File Generator

In [7]:
import pandas as pd
train_df = pd.read_csv('input/llm_spine_parse_train_csv.csv')
train_df.head(10)

,title,file,call_no,author
0,NEWLY INDUSTRIALIZING COUNTRIES AND T~ POLITIC...,03d4dc70-IMG_2817_spine_000.png,HF 1413 .N497 1988,NaN
1,Fair Trade For All,03d4dc70-IMG_2817_spine_001.png,HF 1413 S85 2005,Stiglitz and Charlton
2,Third World Strategy,03d4dc70-IMG_2817_spine_002.png,HF 1413 T5 1983,Gauhar
3,U.S. FOREIGN POLICY AND THE THIRD WORLD: AGEND...,03d4dc70-IMG_2817_spine_003.png,HF 1413 U5x 1983,Lewis Kallab
4,U.S. FOREIGN POLICY and the THIRD WORLD: AGEND...,03d4dc70-IMG_2817_spine_004.png,HF 1413 U535 1985,NaN
5,Neocolonialism; Methods and Manoeuvr~,03d4dc70-IMG_2817_spine_005.png,HF 1413 V3213,V.VAKHRUSHEV
6,The Developing Economies and the International...,03d4dc70-IMG_2817_spine_006.png,HF 1413 V45,S. Venu
7,VIEWS FROM THE SOUTH,03d4dc70-IMG_2817_spine_007.png,HF 1413 .V53 2000,NaN
8,NaN,03d4dc70-IMG_2817_spine_008.png,HF 1413 W4,NaN
9,World Trade Competition,03d4dc70-IMG_2817_spine_009.png,HF 1413 W67,Center for Strategic and International Studies


In [8]:
import json
import base64
import os
from pathlib import Path

# Take first 100 rows of train_df for training data preparation
sample_df = train_df

# Create training data in the format for multimodal LLM fine-tuning
# Each row will include base64-encoded image data

training_data = []

def image_to_base64(image_path):
    """Convert image file to base64 string"""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

for idx, row in sample_df.iterrows():
    # Extract filename from the 'file' column
    filename = Path(row['file']).name
    image_path = f"img/segments_512/{filename}"
    
    # Check if image exists
    if not os.path.exists(image_path):
        print(f"Warning: {image_path} not found, skipping...")
        continue
    
    # Convert image to base64
    image_base64 = image_to_base64(image_path)
    
    # Create the expected JSON output
    expected_json = {
        "title": row['title'] if pd.notna(row['title']) else None,
        "author": row['author'] if pd.notna(row['author']) else None,
        "call_no": row['call_no'] if pd.notna(row['call_no']) else None
    }
    
    # Create a training example with base64-encoded image (image last for readability)
    training_example = {
        "prompt": "Extract the book information from this spine image and return it as JSON with keys: title, author, call_no.",
        "completion": json.dumps(expected_json),
        "image": f"data:image/jpeg;base64,{image_base64}"
    }
    
    training_data.append(training_example)

# Convert to DataFrame for easy viewing (excluding large base64 strings for readability)
training_data_preview = [{k: v if k != 'image' else f"<base64 image data: {len(v)} chars>" 
                          for k, v in item.items()} for item in training_data]
training_data_df = pd.DataFrame(training_data_preview)

# Save to JSONL format (common format for LLM training)
with open('llm_training_data_multimodal.jsonl', 'w') as f:
    for item in training_data:
        f.write(json.dumps(item) + '\n')

print(f"Created {len(training_data)} training examples")
print("\nFirst 3 examples (image data truncated for display):")
print(training_data_df.head(3))
print(f"\nTraining data saved to 'llm_training_data_multimodal.jsonl'")

Created 2730 training examples

First 3 examples (image data truncated for display):
                                              prompt  \
0  Extract the book information from this spine i...   
1  Extract the book information from this spine i...   
2  Extract the book information from this spine i...   

                                          completion  \
0  {"title": "NEWLY INDUSTRIALIZING COUNTRIES AND...   
1  {"title": "Fair Trade For All", "author": "Sti...   
2  {"title": "Third World Strategy", "author": "G...   

                              image  
0  <base64 image data: 56299 chars>  
1  <base64 image data: 46995 chars>  
2  <base64 image data: 52227 chars>  

Training data saved to 'llm_training_data_multimodal.jsonl'
